In [5]:
import numpy as np
from scipy.io import loadmat
import pandas as pd;
import time;
from IPython.display import display;
import pyvista as pv;
from gravity_forward_numba import VecWerSch_numba;
# %%
# ! # Load EROS <Geometry>
eros_mat = loadmat('EROS.mat');
eros_vf  = eros_mat['eros11272_22540'];  # shape: (nVert + nFace, 3)
nF       = 22540;
nVpF     = eros_vf.shape[0];
nV       = nVpF - nF;
Vert     = eros_vf[:nV, :].astype(np.float64);
Faces    = eros_vf[nV:, :].astype(np.int64);
X1, X2   = Vert[:, 0].min(), Vert[:, 0].max();
Y1, Y2   = Vert[:, 1].min(), Vert[:, 1].max();
Z1, Z2   = Vert[:, 2].min(), Vert[:, 2].max();
print(f'Number of <Vertex> : {nV}');
print(f'Number of <Faces>  : {nF}');
print(f'Bounding box (km)  :\n'
      f'X in [{X1:8.4f}, {X2:8.4f}]\n'
      f'Y in [{Y1:8.4f}, {Y2:8.4f}]\n'
      f'Z in [{Z2:8.4f}, {Z2:8.4f}]');

Number of <Vertex> : 11272
Number of <Faces>  : 22540
Bounding box (km)  :
X in [-17.6345,  15.0963]
Y in [ -8.2627,   8.6053]
Z in [  5.9477,   5.9477]


In [6]:
# ! # Load EROS <Gravity> computed in <Matlab>
eros_grav = loadmat('EROS_Grefs.mat');
V   = eros_grav['dV'];
gx  = eros_grav['gx'];   gy = eros_grav['gy'];   gz = eros_grav['gz'];
Txx = eros_grav['Txx']; Txy = eros_grav['Txy']; Txz = eros_grav['Txz'];
Tyy = eros_grav['Tyy']; Tyz = eros_grav['Tyz']; Tzz = eros_grav['Tzz'];
tc_matlab = eros_grav['time_cost'].item();
print(f'Computation size: {V.shape}');
print(f'tc_matlab: {tc_matlab:.2f} sec');

Computation size: (201, 401)
tc_matlab: 297.00 sec


In [7]:
# ! #  Numpy Vectorized code comutation
rho = 2670.;
xgv = np.linspace(-20., 20., 401);
ygv = np.linspace(-10., 10., 201);
[X2d, Y2d] = np.meshgrid(xgv, ygv);
z0 = -6.3;
Z2d = z0 * np.ones(X2d.shape);
P = np.column_stack((X2d.flatten(), Y2d.flatten(), Z2d.flatten()));
######## * Polyhedron
t1 = time.time();
V_cal, gx_cal, gy_cal, gz_cal, \
Txx_cal, Tyy_cal, Tzz_cal, Txy_cal, Txz_cal, Tyz_cal \
    = VecWerSch_numba(P, Vert, Faces, rho);
tc_np = time.time() - t1;
print(f'Computation size: {X2d.shape}');
print(f'tc_numpy: {tc_np:.2f} sec');

Computation size: (201, 401)
tc_numpy: 5.07 sec


In [8]:
# ! #  Pandas disp stats 
fields = ['V', 'gx', 'gy', 'gz', 'Txx', 'Tyy', 'Tzz', 'Txy', 'Txz', 'Tyz']
V_r, gx_r, gy_r, gz_r, Txx_r, Tyy_r, Tzz_r, Txy_r, Txz_r, Tyz_r = \
    (arr[::1, ::1] for arr in (V, gx, gy, gz, Txx, Tyy, Tzz, Txy, Txz, Tyz))
df_ref = pd.DataFrame({
    name: {'Min': arr.min(), 'Max': arr.max(), 'Mean': arr.mean(), 'Std': arr.std()}
    for name, arr in zip(fields, [V_r, gx_r, gy_r, gz_r, Txx_r, Tyy_r, Tzz_r, Txy_r, Txz_r, Tyz_r])
}).T
df_cal = pd.DataFrame({
    name: {'Min': arr.min(), 'Max': arr.max(), 'Mean': arr.mean(), 'Std': arr.std()}
    for name, arr in zip(fields, [V_cal, gx_cal, gy_cal, gz_cal, 
                                  Txx_cal, Tyy_cal, Tzz_cal, Txy_cal, Txz_cal, Tyz_cal])
}).T
df_diff = df_cal - df_ref

# def styled_table(df, title):
#     return (
#         df.style
#         .format('{:12.6f}')
#         .set_properties(**{'text-align': 'center'})
#         .set_caption(f"<h3>{title}</h3>")
#     )
# display(styled_table(df_ref, "Reference"))
# display(styled_table(df_cal, "Polyhedron"))
# display(styled_table(df_diff, "Difference"))

def print_table(df, title):
    print(f"\n{title}")
    print("=" * len(title))
    print(df.to_string(float_format="{:12.6f}".format))
print_table(df_ref, "Reference")
print_table(df_cal, "Polyhedron")
print_table(df_diff, "Difference")


Reference
             Min          Max         Mean          Std
V      19.996389    51.393046    34.991306     7.831494
gx   -211.260642   181.578525    -0.905034   115.061339
gy   -239.478787   223.434960     0.738218   136.183628
gz     28.852651   529.623007   200.902895   125.059607
Txx  -370.561720   139.529568   -53.246244   104.504888
Tyy  -927.214203   149.978225  -129.729110   230.319474
Tzz   -67.485721  1183.542843   182.975354   290.367928
Txy  -278.508550   149.168308    -2.212250    94.837089
Txz  -437.277931   314.923381    -1.987220   169.265985
Tyz  -688.246521   635.419932     0.448210   288.383086

Polyhedron
             Min          Max         Mean          Std
V      19.996389    51.393046    34.991306     7.831494
gx   -211.260642   181.578525    -0.905034   115.061339
gy   -239.478787   223.434960     0.738218   136.183628
gz     28.852651   529.623007   200.902895   125.059607
Txx  -370.561720   139.529568   -53.246244   104.504888
Tyy  -927.214203   149.97